# Visualization workflows

This notebook shows the new unified `db.plot(...)` workflow on the packaged MARIO test databases.

The same plotting engine supports both a guided path for quick inspection and an advanced path with explicit Plotly Express mappings, including custom palettes.

In [1]:
import mario


## Load the packaged test databases

These are the same small MARIO-readable tables used elsewhere in the documentation and test suite, so the examples are reproducible without external downloads.

In [2]:
iot = mario.load_test("IOT")
sut = mario.load_test("SUT")

INFO Parser: excel reading IOT flows from /path/to/MARIO/mario/test/tables/test_IOT_standard.xlsx.


INFO Parser: state payload ready with 6 canonical blocks.


INFO Parser: excel state ready for IOT.


INFO Metadata: initialized.


INFO Parser: excel reading SUT flows from /path/to/MARIO/mario/test/tables/test_SUT_standard.xlsx.


INFO Parser: state payload ready with 10 canonical blocks.


INFO Parser: excel state ready for SUT.


INFO Metadata: initialized.


## Quick path: plot one matrix with a preset

For a first look at a matrix, pass the matrix name and let MARIO choose a useful Plotly Express mapping with a preset such as `"overview"`.

In [3]:
fig = iot.plot(
    matrix="Z",
    preset="overview",
    auto_open=False,
    path=False,
)
fig

## Inspect the dataframe actually plotted

Set `return_data=True` when you want both the Plotly figure and the dataframe passed to Plotly Express after filtering, aggregation and top-n trimming.

In [4]:
fig, plotted = iot.plot(
    matrix="Z",
    preset=None,
    kind="bar",
    x="Sector_from",
    color="Region_to",
    auto_open=False,
    path=False,
    return_data=True,
)
plotted.head()

,Sector_from,Region_to,Value
0,Agriculture,Reg1,7.367798e+06
1,Agriculture,Reg2,1.048720e+05
2,Industry,Reg1,3.487216e+07
3,Industry,Reg2,6.248561e+05
4,Services,Reg1,4.228452e+07


## Advanced path: explicit Plotly Express mappings

More experienced users can control the chart directly through `x`, `color`, `facet_col`, `animation_frame`, `agg`, `top_n` and the other familiar mappings.

In [5]:
fig = iot.plot(
    matrix="Y",
    kind="bar",
    preset=None,
    x="Sector_from",
    color="Consumption category_to",
    facet_col="Region_to",
    agg="sum",
    top_n=6,
    auto_open=False,
    path=False,
)
fig

## Filter before plotting

Use `filters={...}` to keep only the levels you want before MARIO aggregates the data.

In [6]:
fig = iot.plot(
    matrix="Z",
    kind="bar",
    preset=None,
    x="Sector_from",
    color="Sector_to",
    filters={"Region_from": ["Reg1"], "Region_to": ["Reg1"]},
    top_n=4,
    auto_open=False,
    path=False,
)
fig

## SUT-specific views

The same engine works on split SUT blocks such as `U` and `S`, so you can plot activity and commodity dimensions directly.

In [7]:
fig = sut.plot(
    matrix="U",
    kind="bar",
    preset=None,
    x="Commodity_from",
    color="Activity_to",
    facet_col="Region_to",
    top_n=4,
    auto_open=False,
    path=False,
)
fig

## Plot a prepared dataframe

`db.plot(...)` can also accept a custom dataframe when the quantity to visualize is not one raw MARIO matrix. Here we reuse sectoral GDP.

In [8]:
gdp = iot.GDP(total=False).reset_index()
gdp

,Region,Sector,GDP
0,Reg1,Agriculture,6.016424e+06
1,Reg1,Industry,1.497569e+07
2,Reg1,Services,6.206624e+07
3,Reg2,Agriculture,4.539138e+04
4,Reg2,Industry,3.520592e+05
5,Reg2,Services,1.428250e+06


In [9]:
fig = iot.plot(
    data=gdp,
    kind="treemap",
    preset=None,
    y="GDP",
    path_columns=["Sector"],
    auto_open=False,
    path=False,
)
fig

## Change the default palette

Use `mario.set_palette(...)` to change the default discrete color sequence used by `db.plot(...)`. MARIO ships built-in palettes such as `mario`, `Plotly`, `Safe`, `Vivid`, `Pastel`, `Alphabet`, and `McKinsey`.

In [10]:
mario.set_palette(mario_palettes="Safe")

fig = iot.plot(
    matrix="Z",
    preset="overview",
    auto_open=False,
    path=False,
)
fig

You can also pass an explicit list of colors. The palette is session-wide, so reset it when you want to go back to the MARIO defaults.

In [11]:
mario.set_palette(user_palette=["#16335B", "#2F8F9D", "#E6B325"])

fig = sut.plot(
    matrix="U",
    kind="bar",
    preset=None,
    x="Commodity_from",
    color="Region_to",
    facet_col="Activity_to",
    agg="sum",
    auto_open=False,
    path=False,
)
fig